# Session 5: From QuPath to Python

Yesterday you segmented and classified cells in QuPath. This session does the same kind of analysis in Python,
on the same image, using a small tile so that every step runs in seconds. The results won't be as good as
InstanSeg's, and that's deliberate: building a segmentation from simple steps shows how each choice you make
(a threshold, a distance) changes the cells you find, and in turn the biological conclusions you draw.

| In QuPath (Day 1) | In Python (today) |
|---|---|
| Open an image; check its channels and pixel size | `bioio.BioImage`: `channel_names`, `physical_pixel_sizes` |
| Import GeoJSON annotations and detections | `geopandas.read_file` |
| Cell detection: threshold (one fixed value), split by shape, cell expansion | `skimage`: Gaussian smoothing, automatic thresholds (Otsu, Li, ... as in Fiji), watershed, `expand_labels` |
| Measurement table | `skimage.measure.regionprops_table`, as a pandas `DataFrame` |
| Single-measurement and composite classifiers | comparisons such as `cells["CD8A"] > 20`, combined with `&` and `|` |
| Export detections as GeoJSON | `geopandas.GeoDataFrame.to_file` |

Cells marked **Exercise** are for you to complete. The completed notebook is in `solutions/`.

## 1. Jupyter notebooks in two minutes

A notebook is a list of **cells**. Text cells, like this one, explain; code cells run Python. Click a code cell
and press **Shift + Enter** to run it and move to the next one. The number that appears in `[ ]` beside a cell
shows the order cells ran in.

Variables created in one cell are available in every cell you run after it. If things get confused, use
**Kernel › Restart Kernel and Run All Cells** to start again from the top.

In [ ]:
pixel_size = 0.2125  # µm per pixel
tile_width_px = 941
tile_width_px * pixel_size  # the last line of a cell is displayed

## 2. Open the image you used in QuPath

The Day 1 image is an OME-TIFF: a TIFF file with **metadata** describing the channels, the pixel size and more.
We open it with [bioio](https://bioio-devs.github.io/bioio/), which reads most microscopy formats (CZI, LIF,
ND2, OME-TIFF, ...) through one interface. Its Bio-Formats plugin uses the same library as Fiji and QuPath.

Bio-Formats needs Java, which bioio downloads the first time it's used. If that didn't work on your computer,
the cell below falls back to a reader written purely in Python. It gives the same result; it's just slower on
large images, because it reads whole channels rather than only the part you ask for.

In [ ]:
from pathlib import Path

import bioio_bioformats
import bioio_ome_tiff
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from bioio import BioImage

DATA = Path("../data/qupath")
IMAGE = DATA / "xenium_rcc_crop_if.ome.tif"

try:
    image = BioImage(IMAGE, reader=bioio_bioformats.Reader, dask_tiles=True, tile_size=(512, 512))
    print("Reading with Bio-Formats")
except Exception as error:
    print(f"Bio-Formats isn't available ({type(error).__name__}); reading with bioio-ome-tiff instead")
    image = BioImage(IMAGE, reader=bioio_ome_tiff.Reader)

image

The metadata tells us what's in the file. Xenium images have 35 channels: DAPI, the stains 10x used for
segmentation, and 27 antibody (protein) channels.

In [ ]:
print("Dimensions:", image.dims)
print("Pixel size (µm):", image.physical_pixel_sizes)
print("Channels:", image.channel_names)

PIXEL_SIZE = image.physical_pixel_sizes.X  # µm per pixel

**Questions**

- How big is the image, in pixels and in mm?
- How many values does the image hold in total? At 2 bytes each, how much memory would it need?

Loading all of that would take several GB. Instead, `get_image_dask_data` returns a **lazy** array: it knows
the image's shape, but reads pixels only when you ask for them. We take a 200 µm square tile and five
channels: DAPI for the nuclei, and four markers.

Positions are in pixels of the full image, the same as in QuPath (where you can see them in the bottom-right
corner as you move the cursor).

In [ ]:
TILE_UM = (940, 840, 1140, 1040)  # x0, y0, x1, y1 in µm
x0, y0, x1, y1 = (round(value / PIXEL_SIZE) for value in TILE_UM)  # the same in pixels
print(f"Tile: x {x0}–{x1}, y {y0}–{y1} px")

def read_channel(name):
    """Read one channel of the tile. Nothing is read until .compute()."""
    lazy = image.get_image_dask_data("YX", C=image.channel_names.index(name))  # the whole channel, lazily
    return lazy[y0:y1, x0:x1].compute().astype(np.float32)


CHANNELS = ["DAPI", "CD3E", "CD8A", "CD68", "PanCK"]
channels = {name: read_channel(name) for name in CHANNELS}
print(f"Read {len(channels)} channels of {channels['DAPI'].shape} pixels")

To display images in µm-free, full-image pixel coordinates, we pass `extent` to `imshow`: then outlines from
QuPath, which use the same coordinates, land in the right place.

In [ ]:
EXTENT = [x0, x1, y1, y0]  # left, right, bottom, top


def stretch(channel, low=1, high=99.5):
    """Scale a channel to 0–1 for display, between two percentiles."""
    lo, hi = np.percentile(channel, [low, high])
    return np.clip((channel - lo) / (hi - lo + 1e-9), 0, 1)


fig, axes = plt.subplots(1, len(CHANNELS), figsize=(4 * len(CHANNELS), 4))
for ax, name in zip(axes, CHANNELS):
    ax.imshow(stretch(channels[name]), cmap="gray", extent=EXTENT)
    ax.set_title(name)
    ax.axis("off")
plt.show()

## 3. The QuPath annotations and 10x's cells

GeoJSON is the format QuPath uses to import and export objects. `geopandas` reads it into a table with one row per
object and its outline in the `geometry` column.

GeoJSON files are assumed to be maps, in longitude and latitude, so `geopandas` labels them with that
**coordinate reference system** (CRS). QuPath uses image pixels, not degrees, so we remove the label.
Otherwise plots come out distorted, or fail with "aspect must be finite and positive".

In [ ]:
def read_geojson(path):
    """Read QuPath GeoJSON, whose coordinates are image pixels."""
    return gpd.read_file(path).set_crs(None, allow_override=True)  # remove the longitude/latitude label


annotations = read_geojson(DATA / "xenium_rcc_crop_annotations.geojson")
cells_10x = read_geojson(DATA / "xenium_rcc_crop_10x_cells.geojson")
print(f"{len(annotations)} annotations, {len(cells_10x):,} 10x cells")
annotations[["objectType", "name", "geometry"]]

`.cx[...]` selects the objects in a box, here the tile. We draw DAPI with 10x's cell outlines on top.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(stretch(channels["DAPI"]), cmap="gray", extent=EXTENT)
cells_10x.cx[x0:x1, y0:y1].boundary.plot(ax=ax, color="yellow", linewidth=0.8)
ax.set(xlim=(x0, x1), ylim=(y1, y0), title="DAPI and 10x's cells")
plt.show()

## 4. Find the nuclei

QuPath's cell detection finds nuclei in four steps, which we'll rebuild one at a time:

1. **smooth** the DAPI image, to reduce noise;
2. **threshold**: every pixel brighter than a threshold is "nucleus";
3. **split** touching nuclei;
4. **remove** objects too small to be nuclei.

### Smoothing and the histogram

A threshold is easiest to understand from the histogram of pixel values. Nuclei are the bright pixels, on the
right; background is the large peak on the left. (The y axis is logarithmic, so the rare bright values show.)

In [ ]:
from skimage import filters

dapi = filters.gaussian(channels["DAPI"], sigma=1.0, preserve_range=True)  # sigma in pixels

fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(dapi.ravel(), bins=200, log=True)
ax.set(xlabel="DAPI intensity (smoothed)", ylabel="pixels")
plt.show()

### Automatic thresholds

In QuPath's cell detection you type in one fixed threshold. In the introductory workshop you met the alternative:
let an **algorithm** choose the threshold from the image's histogram, as Fiji's *Image › Adjust › Auto Threshold*
does. scikit-image has the same methods, and `try_all_threshold` shows them side by side, like Fiji's "Try all".

In [ ]:
fig, axes = filters.try_all_threshold(dapi, figsize=(10, 12), verbose=False)
plt.show()

Each method makes a different assumption about the histogram, so each gives a different value. Here are the
values, marked on the histogram.

In [ ]:
METHODS = {
    "Otsu": filters.threshold_otsu,
    "Li": filters.threshold_li,
    "Yen": filters.threshold_yen,
    "Triangle": filters.threshold_triangle,
    "Isodata": filters.threshold_isodata,
    "Mean": filters.threshold_mean,
}
thresholds = {name: float(method(dapi)) for name, method in METHODS.items()}

fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(dapi.ravel(), bins=200, log=True, color="lightgrey")
for (name, value), colour in zip(thresholds.items(), plt.cm.tab10.colors):
    ax.axvline(value, color=colour, label=f"{name}: {value:.0f}")
ax.set(xlabel="DAPI intensity (smoothed)", ylabel="pixels")
ax.legend(fontsize="small")
plt.show()

If the slider below works in your browser, drag it to see how the mask changes with the threshold. (If it doesn't
appear, carry on: nothing later depends on it.)

In [ ]:
from ipywidgets import IntSlider, interact


@interact(threshold=IntSlider(value=round(thresholds["Otsu"]), min=5, max=300, step=5, continuous_update=False))
def show_threshold(threshold):
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    axes[0].imshow(stretch(dapi), cmap="gray", extent=EXTENT)
    axes[1].imshow(dapi > threshold, cmap="gray", extent=EXTENT)
    axes[1].set_title(f"DAPI > {threshold}: {np.mean(dapi > threshold):.0%} of pixels")
    plt.show()

### Splitting touching nuclei

A threshold gives one connected blob wherever nuclei touch. QuPath's "split by shape" separates them, and the usual
way to do the same in Python is a **watershed** on the **distance transform**:

- the distance transform gives each mask pixel its distance to the background, so each nucleus becomes a hill
  with its peak near the centre;
- each peak (at least `min_distance` apart) seeds one nucleus;
- the watershed grows the seeds downhill until they meet, which draws the dividing lines.

Finally, objects smaller than a minimum area are removed, as with QuPath's minimum nucleus area. We put the
steps in a **function**, so we can rerun them with different settings later.

In [ ]:
from scipy import ndimage
from skimage import feature, measure, morphology, segmentation


def find_nuclei(dapi, threshold, min_distance_um=2.5, min_area_um2=10):
    """Threshold the smoothed DAPI image and split touching nuclei. Returns a label image: 0 is background, and
    each nucleus has its own number."""
    mask = morphology.remove_small_holes(dapi > threshold, max_size=int(min_area_um2 / PIXEL_SIZE**2))
    distance = ndimage.distance_transform_edt(mask)
    peaks = feature.peak_local_max(distance, min_distance=round(min_distance_um / PIXEL_SIZE), labels=measure.label(mask))
    seeds = np.zeros(mask.shape, dtype=np.int32)
    seeds[tuple(peaks.T)] = np.arange(1, len(peaks) + 1)
    labels = segmentation.watershed(-distance, seeds, mask=mask)
    labels = morphology.remove_small_objects(labels, max_size=int(min_area_um2 / PIXEL_SIZE**2))
    return segmentation.relabel_sequential(labels)[0]

**Exercise 1.** Run `find_nuclei` with the threshold from each method in `thresholds`, and compare: how many nuclei
does each find, and what do they look like? Show the outlines for each method (`segmentation.mark_boundaries`
draws them onto an image).

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
for ax, (name, value) in zip(axes.flat, thresholds.items()):
    labels = find_nuclei(dapi, value)
    ax.imshow(segmentation.mark_boundaries(stretch(dapi), labels, color=(1, 1, 0)), extent=EXTENT)
    ax.set_title(f"{name}: threshold {value:.0f}, {labels.max()} nuclei")
    ax.axis("off")
plt.show()

**Questions**

- Which method would you choose for this tile? Why?
- The tumour nuclei here are large and dim, and the lymphocyte nuclei small and bright. Can any single threshold
  suit both?

Choose a method in the cell below; the rest of the notebook uses it.

In [ ]:
METHOD = "Otsu"  # change this to your choice
threshold = thresholds[METHOD]
nuclei = find_nuclei(dapi, threshold)
print(f"{METHOD}: threshold {threshold:.0f}, {nuclei.max()} nuclei")

### One image, or many?

So far we have looked at one 200 µm tile. The whole Day 1 image is 1.5 mm across, and it isn't really one image:
the Xenium instrument imaged the section as a grid of overlapping **fields of view** (FOVs), each about
630 × 750 µm, and stitched them together. The grid is in `xenium_rcc_crop_fovs.geojson` (you can import it into
QuPath too).

In the introductory workshop you applied an automatic method to **each image separately**, getting a different
value for each. Here, the equivalent is one threshold per FOV. Let's compare that with one threshold for the whole
stitched image. First, read DAPI for the whole image (one channel, about 100 MB).

In [ ]:
fovs = read_geojson(DATA / "xenium_rcc_crop_fovs.geojson")
whole_dapi = filters.gaussian(image.get_image_dask_data("YX", C=image.channel_names.index("DAPI")).compute().astype(np.float32),
                              sigma=1.0, preserve_range=True)
height, width = whole_dapi.shape

whole_threshold = METHODS[METHOD](whole_dapi)
for i, fov in fovs.iterrows():
    left, top, right, bottom = (round(v) for v in fov.geometry.bounds)
    fovs.loc[i, "threshold"] = METHODS[METHOD](whole_dapi[top:bottom, left:right])
print(f"{METHOD} on the whole image: {whole_threshold:.0f}")
fovs[["name", "threshold"]].round(0)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))
step = 8  # show every 8th pixel: plenty for an overview
ax.imshow(stretch(whole_dapi[::step, ::step]), cmap="gray", extent=[0, width, height, 0])
fovs.boundary.plot(ax=ax, color="cyan", linewidth=1)
for _, fov in fovs.iterrows():
    centre = fov.geometry.centroid
    ax.annotate(f"{fov['name']}\n{fov['threshold']:.0f}", (centre.x, centre.y), color="cyan", ha="center", va="center")
ax.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, color="red", linewidth=2))
ax.set_title(f"{METHOD} threshold for each field of view (whole image: {whole_threshold:.0f}); our tile in red")
plt.show()

Our tile lies in one FOV. Compare the nuclei found with the whole-image threshold and with that FOV's own
threshold.

In [ ]:
tile_fov = fovs[fovs.contains(gpd.points_from_xy([(x0 + x1) / 2], [(y0 + y1) / 2])[0])].iloc[0]
choices = {"whole image": whole_threshold, f"FOV {tile_fov['name']}": tile_fov["threshold"], "this tile": threshold}

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, (name, value) in zip(axes, choices.items()):
    labels = find_nuclei(dapi, value)
    ax.imshow(segmentation.mark_boundaries(stretch(dapi), labels, color=(1, 1, 0)), extent=EXTENT)
    ax.set_title(f"Threshold from {name}: {value:.0f}, {labels.max()} nuclei")
    ax.axis("off")
plt.show()

**Questions**

- Which FOVs get the highest thresholds? Look at what tissue they contain.
- A nucleus near a seam between two FOVs with different thresholds could be found in one and missed in the other.
  Would you be able to tell from your results?
- A threshold per FOV assumes every FOV contains both nuclei and background. What happens in a FOV that is mostly
  empty, or entirely tumour? (The thin FOVs along the bottom edge are only 54 µm tall.)

Here the differences between FOVs come mainly from the tissue in them, because 10x's software has already
corrected the images for uneven illumination. On uncorrected microscope tiles there is a second cause: each tile
can be brighter in the middle than at the edges. Either way, **no single number suits every part of a large
image**, however carefully it was chosen.

**Local (adaptive) thresholding** takes this to its limit: every pixel gets its own threshold from its
neighbourhood. It copes with uneven brightness, but in a neighbourhood with no nuclei it still finds "nuclei" in
the background. Try it on the tile:

In [ ]:
local = filters.threshold_local(dapi, block_size=2 * round(10 / PIXEL_SIZE) + 1)  # a 20 µm neighbourhood
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(dapi > threshold, cmap="gray", extent=EXTENT)
axes[0].set_title(f"One threshold ({METHOD}: {threshold:.0f})")
axes[1].imshow(dapi > local, cmap="gray", extent=EXTENT)
axes[1].set_title("Local threshold (20 µm neighbourhood)")
plt.show()

Deep-learning models such as InstanSeg avoid choosing a threshold altogether: they normalise each patch they see
and have learned what nuclei look like. We'll compare them with our threshold-based nuclei at the end.

## 5. From nuclei to cells, and measurements

Most markers are on the membrane or in the cytoplasm, not the nucleus, so we need cell outlines. Like QuPath's
**cell expansion**, `expand_labels` grows each nucleus outward by a fixed distance, stopping where it meets a
neighbour.

In [ ]:
EXPANSION_UM = 3


def nuclei_to_cells(nuclei, expansion_um):
    return segmentation.expand_labels(nuclei, distance=round(expansion_um / PIXEL_SIZE))


cells = nuclei_to_cells(nuclei, EXPANSION_UM)

fig, ax = plt.subplots(figsize=(8, 8))
composite = np.stack([stretch(channels["CD8A"]), stretch(channels["CD3E"]), stretch(channels["DAPI"])], axis=-1)
ax.imshow(segmentation.mark_boundaries(composite, cells, color=(1, 1, 1)), extent=EXTENT)
ax.set_title("CD8A (red), CD3E (green), DAPI (blue), with cell outlines")
plt.show()

`regionprops_table` measures every object at once, like QuPath's measurement table: here each cell's area,
centre and mean intensity in each channel. A pandas `DataFrame` holds the result, one row per cell.

In [ ]:
def measure_cells(cells):
    """One row per cell: area (µm²), centroid (full-image pixels) and the mean intensity of each channel."""
    table = pd.DataFrame(measure.regionprops_table(
        cells,
        intensity_image=np.stack([channels[name] for name in CHANNELS], axis=-1),
        properties=("label", "area", "centroid", "intensity_mean"),
    ))
    table = table.rename(columns={f"intensity_mean-{i}": name for i, name in enumerate(CHANNELS)})
    table["area"] *= PIXEL_SIZE**2
    table["x"] = table.pop("centroid-1") + x0
    table["y"] = table.pop("centroid-0") + y0
    return table.set_index("label")


measurements = measure_cells(cells)
measurements.describe().round(1)

## 6. Classify the cells

QuPath's **single-measurement classifier** labels a cell positive for a marker when one measurement is above a
threshold. In pandas, a comparison gives a True/False column. Choose each threshold from the marker's histogram:
negative cells form the large peak near 0, and positive cells the long tail.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 3))
for ax, name in zip(axes, ["CD3E", "CD8A", "CD68", "PanCK"]):
    ax.hist(measurements[name], bins=50, log=True)
    ax.set(xlabel=f"mean {name} per cell", ylabel="cells")
plt.show()

**Exercise 2.** Fill in a threshold for each marker in `MARKER_THRESHOLDS`, using the histograms. Then check
your choices: cells that are positive should look positive in the image.

In [ ]:
MARKER_THRESHOLDS = {"CD3E": 20, "CD8A": 15, "CD68": 20, "PanCK": 15}

positive = pd.DataFrame({name: measurements[name] > value for name, value in MARKER_THRESHOLDS.items()})
positive.mean().round(2)  # the fraction of cells positive for each marker

A **composite classifier** combines single markers into phenotypes. `&` means "and", `~` means "not". The order
matters: a cell gets the first phenotype that applies.

In [ ]:
def classify(positive):
    """Phenotype each cell from its positive/negative calls."""
    rules = [
        ("CD8+ T cell", positive["CD3E"] & positive["CD8A"]),
        ("Other T cell", positive["CD3E"] & ~positive["CD8A"]),
        ("Macrophage", positive["CD68"] & ~positive["CD3E"]),
        ("PanCK+", positive["PanCK"] & ~positive["CD3E"] & ~positive["CD68"]),
    ]
    phenotype = pd.Series("Other", index=positive.index)
    for name, rule in reversed(rules):  # earlier rules overwrite later ones, so they win
        phenotype[rule] = name
    return phenotype


COLOURS = {"CD8+ T cell": "yellow", "Other T cell": "lime", "Macrophage": "magenta", "PanCK+": "red", "Other": "grey"}

measurements["phenotype"] = classify(positive)
measurements["phenotype"].value_counts()

In [ ]:
def show_phenotypes(cells, phenotypes, ax, title):
    """Fill each cell with its phenotype's colour, over DAPI."""
    ax.imshow(stretch(channels["DAPI"]), cmap="gray", extent=EXTENT)
    for name, colour in COLOURS.items():
        mask = np.isin(cells, phenotypes.index[phenotypes == name])
        ax.imshow(np.ma.masked_where(~mask, mask), cmap=plt.matplotlib.colors.ListedColormap([colour]),
                  alpha=0.5, extent=EXTENT)
        ax.plot([], [], "s", color=colour, label=name)
    ax.legend(loc="lower right", fontsize="small")
    ax.set_title(title)


fig, axes = plt.subplots(1, 2, figsize=(16, 8))
axes[0].imshow(composite, extent=EXTENT)
axes[0].set_title("CD8A (red), CD3E (green), DAPI (blue)")
show_phenotypes(cells, measurements["phenotype"], axes[1], "Phenotypes")
plt.show()

## 7. Exercise: how much do the segmentation choices matter?

You chose a threshold method and an expansion distance, and other people in the room will have chosen differently.
Does it matter? The function below runs the whole analysis for one pair of choices, keeping the marker
thresholds fixed, and reports what you might put in a paper.

In [ ]:
def analyse(threshold, expansion_um):
    nuclei = find_nuclei(dapi, threshold)
    cells = nuclei_to_cells(nuclei, expansion_um)
    table = measure_cells(cells)
    positive = pd.DataFrame({name: table[name] > value for name, value in MARKER_THRESHOLDS.items()})
    phenotype = classify(positive)
    return {
        "threshold": threshold,
        "expansion (µm)": expansion_um,
        "cells": len(table),
        "median cell area (µm²)": table["area"].median(),
        "CD8+ T cells (%)": 100 * (phenotype == "CD8+ T cell").mean(),
        "PanCK+ (%)": 100 * (phenotype == "PanCK+").mean(),
    }


analyse(threshold, EXPANSION_UM)

**Exercise 3.** Run `analyse` for every threshold method in `thresholds` (plus the whole-image threshold,
`whole_threshold`) with your expansion, and for several expansion distances (for example 0, 1, 2, 3, 5 and 8 µm)
with your method. Collect the results in a `DataFrame`, and plot how the number of cells and the percentage of
CD8+ T cells change.

Hint: a list of dictionaries becomes a table with `pd.DataFrame(list_of_dicts)`.

In [ ]:
choices = thresholds | {"Whole image": whole_threshold}
by_method = pd.DataFrame([{"method": name} | analyse(value, EXPANSION_UM) for name, value in choices.items()])
by_expansion = pd.DataFrame([analyse(threshold, expansion) for expansion in (0, 1, 2, 3, 5, 8)])

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
axes[0].bar(by_method["method"], by_method["cells"])
axes[0].set(ylabel="cells", title=f"Threshold method ({EXPANSION_UM} µm expansion)")
axes[1].bar(by_method["method"], by_method["CD8+ T cells (%)"])
axes[1].set(ylabel="CD8+ T cells (%)", title="Threshold method")
axes[2].plot(by_expansion["expansion (µm)"], by_expansion["CD8+ T cells (%)"], "o-")
axes[2].set(xlabel="cell expansion (µm)", ylabel="CD8+ T cells (%)", title=f"Expansion ({METHOD} threshold)")
for ax in axes[:2]:
    ax.tick_params(axis="x", rotation=45)
plt.show()
by_method.round(1)

In [ ]:
by_expansion.round(1)

**Questions**

- Which choice changes the percentage of CD8+ T cells more: the threshold method or the expansion?
- Why would a larger expansion change the proportion of a phenotype, when it doesn't change the number of cells?
- If two labs analysed this image with different (and equally defensible) choices, could they reach different
  conclusions?

## 8. Same classifier, other segmentations

Keep the marker thresholds fixed and change only the segmentation. First, 10x's cells, from the GeoJSON we loaded
earlier: `rasterio`-style "burning" of the outlines into a label image lets us reuse `measure_cells` unchanged.

In [ ]:
from skimage.draw import polygon as fill_polygon


def outlines_to_labels(outlines):
    """Draw outlines (full-image pixel coordinates) into a label image of the tile."""
    labels = np.zeros(dapi.shape, dtype=np.int32)
    for number, geometry in enumerate(outlines.geometry, start=1):
        for part in getattr(geometry, "geoms", [geometry]):
            x, y = np.asarray(part.exterior.coords).T
            rows, cols = fill_polygon(y - y0, x - x0, shape=labels.shape)
            labels[rows, cols] = number
    return labels


def phenotype_percentages(cells):
    table = measure_cells(cells)
    positive = pd.DataFrame({name: table[name] > value for name, value in MARKER_THRESHOLDS.items()})
    return classify(positive), table


cells_10x_tile = outlines_to_labels(cells_10x.cx[x0:x1, y0:y1])
phenotypes_10x, _ = phenotype_percentages(cells_10x_tile)

Second, InstanSeg, the deep-learning model you used in QuPath, given DAPI and 10x's boundary stain. It takes a
few seconds on a laptop. (Session 6 runs it properly, on a larger region.)

In [ ]:
from instanseg import InstanSeg

boundary = read_channel("ATP1A1/CD45/E-Cadherin")
model = InstanSeg("fluorescence_nuclei_and_cells", verbosity=0)
labels, _ = model.eval_small_image(np.stack([channels["DAPI"], boundary], axis=-1), pixel_size=PIXEL_SIZE)
cells_instanseg = np.asarray(labels.cpu()).squeeze()[1].astype(np.int32)  # [0] is nuclei, [1] cells
phenotypes_instanseg, _ = phenotype_percentages(cells_instanseg)

In [ ]:
comparison = pd.DataFrame({
    "Threshold + watershed": measurements["phenotype"].value_counts(normalize=True),
    "10x": phenotypes_10x.value_counts(normalize=True),
    "InstanSeg": phenotypes_instanseg.value_counts(normalize=True),
}).fillna(0).mul(100).round(1)
comparison.loc["cells"] = [len(measurements), len(phenotypes_10x), len(phenotypes_instanseg)]
comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(24, 8))
show_phenotypes(cells, measurements["phenotype"], axes[0], "Threshold + watershed")
show_phenotypes(cells_10x_tile, phenotypes_10x, axes[1], "10x")
show_phenotypes(cells_instanseg, phenotypes_instanseg, axes[2], "InstanSeg")
plt.show()

**Questions**

- Which segmentation would you trust most for the percentage of CD8+ T cells? Why?
- The classifier is identical in all three. What does that tell you about comparing results between studies?

## 9. Back to QuPath

To check your cells in QuPath, convert the label image to outlines and save them as GeoJSON **detections**. Each
needs a classification with a name and a colour. QuPath uses the same full-image pixel coordinates, so we add the
tile's position.

In [ ]:
from shapely.geometry import Polygon

QUPATH_COLOURS = {"CD8+ T cell": [255, 255, 0], "Other T cell": [0, 255, 0], "Macrophage": [255, 0, 255],
                  "PanCK+": [255, 0, 0], "Other": [128, 128, 128]}


def labels_to_outlines(cells, phenotypes):
    """Trace each cell's outline and return QuPath detections, in full-image pixel coordinates."""
    rows = []
    for region in measure.regionprops(cells):
        top, left, _, _ = region.bbox
        padded = np.pad(region.image, 1)  # so outlines at the tile's edge close
        contour = max(measure.find_contours(padded.astype(float), 0.5), key=len)
        rows.append({
            "objectType": "detection",
            "classification": {"name": phenotypes[region.label], "color": QUPATH_COLOURS[phenotypes[region.label]]},
            "geometry": Polygon(contour[:, ::-1] - 1 + [left + x0, top + y0]),  # (row, col) to (x, y)
        })
    return gpd.GeoDataFrame(rows, geometry="geometry")


detections = labels_to_outlines(cells, measurements["phenotype"])
detections.to_file("../data/session5_cells.geojson", driver="GeoJSON")
print(f"Saved {len(detections)} cells. In QuPath: File › Import objects from file, then choose session5_cells.geojson")

fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(stretch(channels["DAPI"]), cmap="gray", extent=EXTENT)
detections.boundary.plot(ax=ax, color=detections["classification"].map(lambda c: np.array(c["color"]) / 255),
                         linewidth=0.8)
ax.set(xlim=(x0, x1), ylim=(y1, y0), title="The saved outlines, read back as geometry")
plt.show()

## Summary

- The Day 1 image is an ordinary OME-TIFF: bioio reads its metadata and, lazily, just the part you need.
- A classic segmentation is a chain of simple steps (smooth, threshold, split, filter, expand), each with a
  setting you choose.
- Automatic threshold methods disagree with each other, and a large stitched image rarely suits one threshold,
  however it was chosen. Thresholding each field of view separately helps, but brings seams and empty tiles.
- Those choices change not only the outlines but the numbers you would report, even with the classifier held
  fixed.
- QuPath and Python share GeoJSON and pixel coordinates, so objects move freely between them.

In session 6 we bring in the transcripts, which need a richer container than a TIFF, and segment with InstanSeg
properly.